## Cross-Correlation of Two Unit Pulses

Cross-correlation measures the similarity between two signals as one is shifted relative to the other:

$$
R_{xy}(\tau) = \int_{-\infty}^{\infty} x(t+\tau)\, y^*(t)\, dt
$$

In this interactive example:

- **Left panel:** shows two unit pulses, $x(t)$ and $y(t)$, with independent amplitude, width, and time shift.  
- **Right panel:** shows the cross-correlation function $R_{xy}(\tau)$.  
- Sliders allow you to adjust **amplitude**, **width**, and **time shift** of each pulse.  
- The current $\tau$ is highlighted on the cross-correlation curve, showing how similarity changes as $y(t)$ moves relative to $x(t)$.


In [19]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Time axis
fs = 5000
t = np.linspace(-0.05, 0.05, int(fs*0.1), endpoint=False)

# Unit pulse function
def unit_pulse(t, width=0.004, A=1.0):
    return A * np.where((t >= 0) & (t <= width), 1.0, 0.0)

def crosscorr_both(x, y, t):
    dt = t[1] - t[0]
    R_xy = np.correlate(x, y, mode='full') * dt
    R_tilde = np.correlate(y, x, mode='full') * dt
    tau = np.linspace(-t[-1]-dt, t[-1]+dt, len(R_xy))
    return tau, R_xy, R_tilde

def compute_phase_latency(x, y, fs):
    # FFT
    X = np.fft.fft(x)
    Y = np.fft.fft(y)
    
    freqs = np.fft.fftfreq(len(x), 1/fs)
    
    # Take only positive frequencies
    pos = freqs > 0
    freqs = freqs[pos]
    X = X[pos]
    Y = Y[pos]
    
    # Find dominant frequency component
    idx = np.argmax(np.abs(X))
    
    phase_x = np.angle(X[idx])
    phase_y = np.angle(Y[idx])
    
    phase_diff = np.unwrap([phase_x, phase_y])[1] - np.unwrap([phase_x, phase_y])[0]
    
    f0 = freqs[idx]
    
    # latency from phase difference
    latency = -phase_diff / (2*np.pi*f0)
    
    return f0, phase_diff, latency

def plot_crosscorr(A1=1.0, width1=0.004, tau1=0.0,
                   A2=1.0, width2=0.004, tau2=0.0,
                   tau_slider=0.0):

    x = unit_pulse(t - tau1, width=width1, A=A1)
    y = unit_pulse(t - tau2, width=width2, A=A2)
    
    tau_vals, R_xy, R_tilde = crosscorr_both(x, y, t)
    
    f0, phase_diff, latency = compute_phase_latency(x, y, fs)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # --- LEFT: Signals ---
    ax1 = axes[0]
    ax1.plot(t, x, label='x(t)')
    ax1.plot(t, y, label='y(t)')
    ax1.set_xlabel('Time [s]')
    ax1.set_ylabel('Amplitude')
    ax1.set_title('Signals')
    ax1.grid(True)
    ax1.legend()
    
    # --- RIGHT: Cross-correlation ---
    ax2 = axes[1]
    ax2.plot(tau_vals, R_xy, linewidth=2,
             label=r'$R_{xy}(\tau)$')
    ax2.plot(tau_vals, R_tilde, '--', linewidth=2,
             label=r'$\tilde{R}_{xy}(\tau)$')
    ax2.axvline(tau_slider, color='k', linestyle=':', linewidth=1)
    ax2.set_xlabel('τ [s]')
    ax2.set_ylabel('Cross-correlation')
    ax2.set_title('Cross-Correlation')
    ax2.grid(True)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"Dominant frequency: {f0:.2f} Hz")
    print(f"Phase difference: {phase_diff:.3f} rad")
    print(f"Estimated latency: {latency*1000:.3f} ms")

interact(
    plot_crosscorr,
    A1=FloatSlider(min=0.1, max=2.0, step=0.1, value=1.0, description='A1'),
    width1=FloatSlider(min=0.001, max=0.05, step=0.001, value=0.01, description='Width1'),
    tau1=FloatSlider(min=-0.02, max=0.02, step=0.0005, value=0.0, description='τ1'),
    A2=FloatSlider(min=0.1, max=2.0, step=0.1, value=1.0, description='A2'),
    width2=FloatSlider(min=0.001, max=0.05, step=0.001, value=0.03, description='Width2'),
    tau2=FloatSlider(min=-0.02, max=0.02, step=0.0005, value=0.0, description='τ2'),
    tau_slider=FloatSlider(min=-0.05, max=0.05, step=0.0005, value=0.0, description='τ')
)


interactive(children=(FloatSlider(value=1.0, description='A1', max=2.0, min=0.1), FloatSlider(value=0.01, desc…

<function __main__.plot_crosscorr(A1=1.0, width1=0.004, tau1=0.0, A2=1.0, width2=0.004, tau2=0.0, tau_slider=0.0)>

In [21]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

# Time axis
fs = 5000
t = np.linspace(-0.1, 0.2, int(fs*0.3), endpoint=False)  # extend for multiple pulses

# Multi-pulse signal
def multi_pulse_signal(t, T=0.01, A=1.0, N_pulses=2, tau_shift=0.0):
    sig = np.zeros_like(t)
    for n in range(N_pulses):
        pulse_start = n * 2 * T  # pulses spaced by 2*T
        pulse_end = pulse_start + T
        sig += ((t - tau_shift) >= pulse_start) & ((t - tau_shift) < pulse_end)
    return A * sig

# Cross-correlation
def crosscorr(x, y, t):
    dt = t[1] - t[0]
    R = np.correlate(x, y, mode='full') * dt
    tau = np.linspace(-t[-1]-dt, t[-1]+dt, len(R))
    return tau, R

# Interactive plot
def plot_multi_pulse_xcorr(T=0.01, N_pulses=2, tau_slider=0.0):

    # Original signals (fixed for cross-correlation)
    x = multi_pulse_signal(t, T=T, N_pulses=N_pulses)
    y = multi_pulse_signal(t, T=T, N_pulses=N_pulses)
    
    # Shift x by tau_slider for visualization
    x_shifted = multi_pulse_signal(t, T=T, N_pulses=N_pulses, tau_shift=tau_slider)
    
    # Cross-correlation
    tau_vals, R_xy = crosscorr(x, y, t)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # --- LEFT: shifted signals ---
    ax1 = axes[0]
    ax1.plot(t, x_shifted, label=f'x(t) shifted by τ={tau_slider:.3f}s')
    ax1.plot(t, y, label='y(t) (fixed)')
    ax1.set_title(f"{N_pulses} Pulses Signals")
    ax1.set_xlabel("Time [s]")
    ax1.set_ylabel("Amplitude")
    ax1.grid(True)
    ax1.legend()
    
    # --- RIGHT: cross-correlation ---
    ax2 = axes[1]
    ax2.plot(tau_vals, R_xy, label='Rxy(τ)')
    ax2.axvline(tau_slider, color='r', linestyle='--', linewidth=1.5, label=f'τ = {tau_slider:.3f}s')
    
    # mark the point
    R_at_tau = np.interp(tau_slider, tau_vals, R_xy)
    ax2.plot(tau_slider, R_at_tau, 'ro')
    
    ax2.set_title("Cross-Correlation")
    ax2.set_xlabel("τ [s]")
    ax2.set_ylabel("Rxy(τ)")
    ax2.grid(True)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"Cross-correlation at τ = {tau_slider:.4f} s: Rxy = {R_at_tau:.5f}")

# Interactive widget
interact(
    plot_multi_pulse_xcorr,
    T=FloatSlider(min=0.005, max=0.03, step=0.001, value=0.01, description='Pulse Width T'),
    N_pulses=IntSlider(min=1, max=10, step=1, value=2, description='Number of Pulses'),
    tau_slider=FloatSlider(min=-0.05, max=0.2, step=0.001, value=0.0, description='τ (shift)')
)


interactive(children=(FloatSlider(value=0.01, description='Pulse Width T', max=0.03, min=0.005, step=0.001), I…

<function __main__.plot_multi_pulse_xcorr(T=0.01, N_pulses=2, tau_slider=0.0)>

In [25]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def periodic_autocorr_demo(tau_shift=0.0):

    # -------------------------
    # Parameters
    # -------------------------
    A = 1.0
    T0 = 1.0
    T = 3*T0
    fs = 2000
    periods = 2
    
    t = np.arange(0, periods*T, 1/fs)
    N = len(t)

    # -------------------------
    # Periodic signal
    # -------------------------
    t_mod = np.mod(t, T)
    x = np.where(t_mod < 2*T0, A, -A)

    # -------------------------
    # Circular shift
    # -------------------------
    shift_samples = int(np.round(tau_shift * fs))
    x_shifted = np.roll(x, -shift_samples)

    # Product (this is what gets integrated!)
    product = x * x_shifted

    # -------------------------
    # Periodic autocorrelation
    # -------------------------
    X = np.fft.fft(x)
    R = np.fft.ifft(np.abs(X)**2).real / N
    R = np.fft.fftshift(R)

    tau = np.arange(-N//2, N//2) / fs
    idx = (np.abs(tau - tau_shift)).argmin()
    R_tau = R[idx]

    # -------------------------
    # Plot
    # -------------------------
    plt.figure(figsize=(12,5))

    # LEFT PANEL
    plt.subplot(1,2,1)
    plt.plot(t, x, label='x(t)', linewidth=2)
    plt.plot(t, x_shifted, '--', label='x(t+τ)', linewidth=2)

    # Fill positive overlap (A*A)
    plt.fill_between(t, 0, product,
                     where=(product > 0),
                     alpha=0.3)

    # Fill negative overlap (-A*A)
    plt.fill_between(t, 0, product,
                     where=(product < 0),
                     alpha=0.3)

    plt.title(f'Overlap Visualization (τ = {tau_shift:.3f})')
    plt.xlabel('Time')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.grid(True)

    # RIGHT PANEL
    plt.subplot(1,2,2)
    plt.plot(tau, R, linewidth=2)
    plt.scatter(tau[idx], R_tau, s=100)
    plt.title('Periodic Power Autocorrelation')
    plt.xlabel('Lag τ')
    plt.ylabel('Rxx(τ)')
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# Slider
interact(periodic_autocorr_demo,
         tau_shift=FloatSlider(min=-3, max=3, step=0.01, value=0));


interactive(children=(FloatSlider(value=0.0, description='tau_shift', max=3.0, min=-3.0, step=0.01), Output())…